# Task 3 - Aprendizaje Supervisado

Este notebook funciona como registro incremental de experimentos supervisados para predecir `condition`. La metrica principal sera `recall_condition_1`, porque en un escenario medico interesa reducir falsos negativos: pacientes con condicion real que el modelo clasifica como sanos.

## Estrategia de validacion

- Dataset base: `data/processed/foot_clean.csv`.
- Test final estratificado: 20% reservado hasta el cierre del experimento.
- Validacion interna: `RepeatedStratifiedKFold`, 5 folds y 30 repeticiones.
- Preprocesamiento dentro de `Pipeline`: estandarizacion de continuas, one-hot encoding de discretas y paso directo de binarias.
- Coste medico inicial: `FN=5`, `FP=1`, normalizado por paciente.

In [6]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd / "mundial_project" / "src").exists():
    PROJECT_ROOT = cwd / "mundial_project"
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError(f"No se encontro la carpeta src desde {cwd}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

PROJECT_ROOT

WindowsPath('E:/University/4to/ML/mundial_project')

## Experimento 1 - Regresion Logistica

La regresion logistica se usa como primera linea base porque entrega probabilidades, es interpretable y permite contrastar rapidamente si una frontera lineal captura senales utiles para `condition`.

In [7]:
from task3_logistic_regression import run

result = run()
result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\logistic_regression
               metric     mean      std      min      max
             accuracy 0.849770 0.050816 0.723404 0.958333
precision_condition_1 0.857264 0.065266 0.680000 1.000000
   recall_condition_1 0.813521 0.085339 0.590909 1.000000
       f1_condition_1 0.831736 0.059171 0.684211 0.954545
              roc_auc 0.915978 0.035063 0.789091 0.989091
         medical_cost 0.493505 0.198389 0.085106 1.021277
      false_negatives 4.066667 1.866938 0.000000 9.000000
      false_positives 3.053333 1.633486 0.000000 8.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.816667               0.793103            0.821429        0.807018      0.516667 0.896205            26.0              6

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/logistic_regression')

In [8]:
cv_summary = pd.read_csv(result["output_dir"] / "cv_metricas_resumen.csv")
cv_summary

,metric,mean,std,min,max
0,accuracy,0.849770,0.050816,0.723404,0.958333
1,precision_condition_1,0.857264,0.065266,0.680000,1.000000
2,recall_condition_1,0.813521,0.085339,0.590909,1.000000
3,f1_condition_1,0.831736,0.059171,0.684211,0.954545
4,roc_auc,0.915978,0.035063,0.789091,0.989091
5,medical_cost,0.493505,0.198389,0.085106,1.021277
6,false_negatives,4.066667,1.866938,0.000000,9.000000
7,false_positives,3.053333,1.633486,0.000000,8.000000


In [9]:
test_metrics = pd.read_csv(result["output_dir"] / "test_metricas.csv")
test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.816667,0.793103,0.821429,0.807018,0.516667,0.896205,26.0,6.0,5.0,23.0


## Experimento 2 - Naive Bayes

Naive Bayes se usa como segundo baseline porque es un modelo probabilistico simple, rapido y util para contrastar contra la regresion logistica. En este dataset se usa `GaussianNB` sobre las variables preprocesadas.

In [10]:
import importlib
import sys
import task3_common

importlib.reload(task3_common)
if "task3_naive_bayes" in sys.modules:
    task3_naive_bayes = importlib.reload(sys.modules["task3_naive_bayes"])
else:
    task3_naive_bayes = importlib.import_module("task3_naive_bayes")
from task3_naive_bayes import run as run_naive_bayes

naive_bayes_result = run_naive_bayes()
naive_bayes_result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\naive_bayes
               metric     mean      std      min       max
             accuracy 0.801723 0.072685 0.574468  0.936170
precision_condition_1 0.825382 0.081732 0.548387  1.000000
   recall_condition_1 0.728167 0.149570 0.227273  1.000000
       f1_condition_1 0.764333 0.107612 0.333333  0.936170
              roc_auc 0.883075 0.050640 0.729021  0.992727
         medical_cost 0.698443 0.337073 0.063830  1.872340
      false_negatives 5.926667 3.270976 0.000000 17.000000
      false_positives 3.473333 2.144751 0.000000 14.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.833333               0.821429            0.821429        0.821429           0.5  0.83817            27.0              

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/naive_bayes')

In [11]:
naive_bayes_cv_summary = pd.read_csv(naive_bayes_result["output_dir"] / "cv_metricas_resumen.csv")
naive_bayes_cv_summary

,metric,mean,std,min,max
0,accuracy,0.801723,0.072685,0.574468,0.936170
1,precision_condition_1,0.825382,0.081732,0.548387,1.000000
2,recall_condition_1,0.728167,0.149570,0.227273,1.000000
3,f1_condition_1,0.764333,0.107612,0.333333,0.936170
4,roc_auc,0.883075,0.050640,0.729021,0.992727
5,medical_cost,0.698443,0.337073,0.063830,1.872340
6,false_negatives,5.926667,3.270976,0.000000,17.000000
7,false_positives,3.473333,2.144751,0.000000,14.000000


In [12]:
naive_bayes_test_metrics = pd.read_csv(naive_bayes_result["output_dir"] / "test_metricas.csv")
naive_bayes_test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.833333,0.821429,0.821429,0.821429,0.5,0.83817,27.0,5.0,5.0,23.0


## Experimento 3 - KNN

KNN se usa como modelo basado en instancias. Es sensible a la escala, por eso mantiene el mismo `Pipeline` con estandarizacion y one-hot encoding antes de calcular vecinos cercanos. Esta primera prueba usa `k=5` como baseline antes de optimizar hiperparametros en Task 4.

In [13]:
import importlib
import sys
import task3_common

importlib.reload(task3_common)
if "task3_knn" in sys.modules:
    task3_knn = importlib.reload(sys.modules["task3_knn"])
else:
    task3_knn = importlib.import_module("task3_knn")
from task3_knn import run as run_knn

knn_result = run_knn()
knn_result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\knn
               metric     mean      std      min       max
             accuracy 0.822488 0.049537 0.659574  0.916667
precision_condition_1 0.826025 0.071473 0.650000  1.000000
   recall_condition_1 0.785281 0.076233 0.545455  0.954545
       f1_condition_1 0.802203 0.056089 0.600000  0.909091
              roc_auc 0.880634 0.047671 0.706364  0.972727
         medical_cost 0.572512 0.177437 0.212766  1.191489
      false_negatives 4.680000 1.664150 1.000000 10.000000
      false_positives 3.733333 1.759601 0.000000  9.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.816667               0.793103            0.821429        0.807018      0.516667 0.848772            26.0              6.0     

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/knn')

In [14]:
knn_cv_summary = pd.read_csv(knn_result["output_dir"] / "cv_metricas_resumen.csv")
knn_cv_summary

,metric,mean,std,min,max
0,accuracy,0.822488,0.049537,0.659574,0.916667
1,precision_condition_1,0.826025,0.071473,0.650000,1.000000
2,recall_condition_1,0.785281,0.076233,0.545455,0.954545
3,f1_condition_1,0.802203,0.056089,0.600000,0.909091
4,roc_auc,0.880634,0.047671,0.706364,0.972727
5,medical_cost,0.572512,0.177437,0.212766,1.191489
6,false_negatives,4.680000,1.664150,1.000000,10.000000
7,false_positives,3.733333,1.759601,0.000000,9.000000


In [15]:
knn_test_metrics = pd.read_csv(knn_result["output_dir"] / "test_metricas.csv")
knn_test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.816667,0.793103,0.821429,0.807018,0.516667,0.848772,26.0,6.0,5.0,23.0


## Experimento 4 - Arbol de Decision

El arbol de decision se prueba por su interpretabilidad: permite expresar predicciones como reglas. Esta primera version usa CART con `criterion='gini'`, `max_depth=4` y `min_samples_leaf=5` para reducir overfitting antes de la optimizacion formal de Task 4.

In [16]:
import importlib
import sys
import task3_common

importlib.reload(task3_common)
if "task3_decision_tree" in sys.modules:
    task3_decision_tree = importlib.reload(sys.modules["task3_decision_tree"])
else:
    task3_decision_tree = importlib.import_module("task3_decision_tree")
from task3_decision_tree import run as run_decision_tree

decision_tree_result = run_decision_tree()
decision_tree_result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\decision_tree
               metric     mean      std      min       max
             accuracy 0.776421 0.051282 0.617021  0.895833
precision_condition_1 0.786415 0.081014 0.600000  1.000000
   recall_condition_1 0.719293 0.095358 0.363636  0.954545
       f1_condition_1 0.745791 0.062713 0.484848  0.893617
              roc_auc 0.825273 0.054946 0.608182  0.960664
         medical_cost 0.739938 0.209023 0.187500  1.553191
      false_negatives 6.113333 2.067767 1.000000 14.000000
      false_positives 4.480000 2.135416 0.000000 11.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.733333                 0.6875            0.785714        0.733333      0.666667 0.824777            22.0            

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/decision_tree')

In [17]:
decision_tree_cv_summary = pd.read_csv(decision_tree_result["output_dir"] / "cv_metricas_resumen.csv")
decision_tree_cv_summary

,metric,mean,std,min,max
0,accuracy,0.776421,0.051282,0.617021,0.895833
1,precision_condition_1,0.786415,0.081014,0.600000,1.000000
2,recall_condition_1,0.719293,0.095358,0.363636,0.954545
3,f1_condition_1,0.745791,0.062713,0.484848,0.893617
4,roc_auc,0.825273,0.054946,0.608182,0.960664
5,medical_cost,0.739938,0.209023,0.187500,1.553191
6,false_negatives,6.113333,2.067767,1.000000,14.000000
7,false_positives,4.480000,2.135416,0.000000,11.000000


In [18]:
decision_tree_test_metrics = pd.read_csv(decision_tree_result["output_dir"] / "test_metricas.csv")
decision_tree_test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.733333,0.6875,0.785714,0.733333,0.666667,0.824777,22.0,10.0,6.0,22.0


## Experimento 5 - SVM

SVM se prueba con kernel RBF para capturar fronteras no lineales. Esta primera version usa `C=1`, `gamma='scale'` y `probability=True` para obtener probabilidades comparables en ROC-AUC.

In [19]:
import importlib
import sys
import task3_common

importlib.reload(task3_common)
if "task3_svm" in sys.modules:
    task3_svm = importlib.reload(sys.modules["task3_svm"])
else:
    task3_svm = importlib.import_module("task3_svm")
from task3_svm import run as run_svm

svm_result = run_svm()
svm_result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\svm
               metric     mean      std      min      max
             accuracy 0.838652 0.049499 0.708333 0.957447
precision_condition_1 0.840640 0.066556 0.666667 1.000000
   recall_condition_1 0.806638 0.079749 0.590909 1.000000
       f1_condition_1 0.820524 0.057085 0.666667 0.950000
              roc_auc 0.911232 0.035757 0.792727 0.993007
         medical_cost 0.517021 0.186019 0.085106 1.042553
      false_negatives 4.213333 1.740118 0.000000 9.000000
      false_positives 3.433333 1.631965 0.000000 8.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
      0.8                   0.75            0.857143             0.8      0.466667 0.881696            24.0              8.0              

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/svm')

In [20]:
svm_cv_summary = pd.read_csv(svm_result["output_dir"] / "cv_metricas_resumen.csv")
svm_cv_summary

,metric,mean,std,min,max
0,accuracy,0.838652,0.049499,0.708333,0.957447
1,precision_condition_1,0.840640,0.066556,0.666667,1.000000
2,recall_condition_1,0.806638,0.079749,0.590909,1.000000
3,f1_condition_1,0.820524,0.057085,0.666667,0.950000
4,roc_auc,0.911232,0.035757,0.792727,0.993007
5,medical_cost,0.517021,0.186019,0.085106,1.042553
6,false_negatives,4.213333,1.740118,0.000000,9.000000
7,false_positives,3.433333,1.631965,0.000000,8.000000


In [21]:
svm_test_metrics = pd.read_csv(svm_result["output_dir"] / "test_metricas.csv")
svm_test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.8,0.75,0.857143,0.8,0.466667,0.881696,24.0,8.0,4.0,24.0


## Experimento 6 - Red Neuronal MLP

El MLP se prueba como modelo no lineal flexible. Esta primera version usa una red pequena `(16, 8)` con regularizacion `alpha=0.001` y early stopping. Al ser un dataset pequeno, este experimento tambien sirve para detectar si la red subajusta o es inestable frente a modelos mas simples.

In [22]:
import importlib
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "mundial_project" / "src").exists():
    project_root = cwd / "mundial_project"
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise RuntimeError(f"No se encontro la carpeta src desde {cwd}")
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

import task3_common

importlib.reload(task3_common)
if "task3_mlp" in sys.modules:
    task3_mlp = importlib.reload(sys.modules["task3_mlp"])
else:
    task3_mlp = importlib.import_module("task3_mlp")
from task3_mlp import run as run_mlp

mlp_result = run_mlp()
mlp_result["output_dir"]

c:\Users\Josue\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:175: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


CSV usado: E:\University\4to\ML\mundial_project\data\processed\foot_clean.csv
Resultados guardados en: E:\University\4to\ML\mundial_project\outputs\task3\mlp
               metric     mean      std      min       max
             accuracy 0.732045 0.075504 0.510638  0.914894
precision_condition_1 0.786097 0.094621 0.461538  1.000000
   recall_condition_1 0.572020 0.154867 0.181818  0.952381
       f1_condition_1 0.652368 0.124726 0.266667  0.909091
              roc_auc 0.805088 0.082060 0.567766  0.952727
         medical_cost 1.055083 0.354439 0.234043  2.000000
      false_negatives 9.326667 3.372802 1.000000 18.000000
      false_positives 3.373333 1.603464 0.000000  8.000000
Metricas de test final:
 accuracy  precision_condition_1  recall_condition_1  f1_condition_1  medical_cost  roc_auc  true_negatives  false_positives  false_negatives  true_positives
 0.583333               0.636364                0.25        0.358974      1.816667 0.714286            28.0              4.0     

WindowsPath('E:/University/4to/ML/mundial_project/outputs/task3/mlp')

In [23]:
mlp_cv_summary = pd.read_csv(mlp_result["output_dir"] / "cv_metricas_resumen.csv")
mlp_cv_summary

,metric,mean,std,min,max
0,accuracy,0.732045,0.075504,0.510638,0.914894
1,precision_condition_1,0.786097,0.094621,0.461538,1.000000
2,recall_condition_1,0.572020,0.154867,0.181818,0.952381
3,f1_condition_1,0.652368,0.124726,0.266667,0.909091
4,roc_auc,0.805088,0.082060,0.567766,0.952727
5,medical_cost,1.055083,0.354439,0.234043,2.000000
6,false_negatives,9.326667,3.372802,1.000000,18.000000
7,false_positives,3.373333,1.603464,0.000000,8.000000


In [24]:
mlp_test_metrics = pd.read_csv(mlp_result["output_dir"] / "test_metricas.csv")
mlp_test_metrics

,accuracy,precision_condition_1,recall_condition_1,f1_condition_1,medical_cost,roc_auc,true_negatives,false_positives,false_negatives,true_positives
0,0.583333,0.636364,0.25,0.358974,1.816667,0.714286,28.0,4.0,21.0,7.0


## Comparacion final de modelos

La comparacion consolidada se genera desde `task3_comparacion_modelos.py`. El ranking prioriza `recall_condition_1` en validacion cruzada y usa el coste medico como criterio complementario, porque los falsos negativos son el error mas grave en este contexto.

In [25]:
import importlib
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "mundial_project" / "src").exists():
    project_root = cwd / "mundial_project"
elif (cwd.parent / "src").exists():
    project_root = cwd.parent
else:
    raise RuntimeError(f"No se encontro la carpeta src desde {cwd}")
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

if "task3_comparacion_modelos" in sys.modules:
    task3_comparacion_modelos = importlib.reload(sys.modules["task3_comparacion_modelos"])
else:
    task3_comparacion_modelos = importlib.import_module("task3_comparacion_modelos")

comparison = task3_comparacion_modelos.build_comparison(project_root / "outputs" / "task3")
task3_comparacion_modelos.compact_columns(comparison)

,modelo,cv_recall_condition_1_mean,cv_recall_condition_1_std,cv_roc_auc_mean,cv_medical_cost_mean,cv_false_negatives_mean,test_recall_condition_1,test_roc_auc,test_false_negatives,test_false_positives,test_medical_cost
0,Regresion Logistica,0.813521,0.085339,0.915978,0.493505,4.066667,0.821429,0.896205,5.0,6.0,0.516667
1,SVM RBF,0.806638,0.079749,0.911232,0.517021,4.213333,0.857143,0.881696,4.0,8.0,0.466667
2,KNN,0.785281,0.076233,0.880634,0.572512,4.680000,0.821429,0.848772,5.0,6.0,0.516667
3,Naive Bayes,0.728167,0.149570,0.883075,0.698443,5.926667,0.821429,0.838170,5.0,5.0,0.500000
4,Arbol de Decision,0.719293,0.095358,0.825273,0.739938,6.113333,0.785714,0.824777,6.0,10.0,0.666667
5,Red Neuronal MLP,0.572020,0.154867,0.805088,1.055083,9.326667,0.250000,0.714286,21.0,4.0,1.816667


## Lectura medica inicial

Para decidir si el modelo es aceptable, revisar primero `recall_condition_1`, falsos negativos y coste medico. La accuracy queda como metrica secundaria, porque puede ocultar errores clinicamente mas graves.